In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/Churn_Modelling.csv")
df.head(5)
df.info()
df["Exited"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


Exited
0    7963
1    2037
Name: count, dtype: int64

In [25]:
df.groupby("Exited")["Age"].mean()  #yaş ortalamasına bakılır buna göre bir analiz yapılabilir


Exited
0    37.408389
1    44.837997
Name: Age, dtype: float64

In [26]:
df.groupby("Exited")["IsActiveMember"].mean()  #aktif üyelik oranına bakılır

Exited
0    0.554565
1    0.360825
Name: IsActiveMember, dtype: float64

In [27]:
df.groupby("Geography")["Exited"].mean()  #ülkelere göre churn oranına bakılır buna göre analiz yapılacak

Geography
France     0.161548
Germany    0.324432
Spain      0.166734
Name: Exited, dtype: float64

In [28]:
df_model = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
#kalıcı silmedim çünkü bir hata olursa geri dönebilirim bu şekilde


In [29]:
df_model.dtypes

CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [30]:
df_model["Gender"] = df_model["Gender"].map({
    "Male":1,
    "Female":0
}
)  #gender sütunu için encoding yaptım

df_model["Gender"].value_counts()

Gender
1    5457
0    4543
Name: count, dtype: int64

In [31]:
df_model = pd.get_dummies(
    df_model,
    columns=["Geography"],
    drop_first=True
)

#one hot encoding yaptım geography sütunu için

In [32]:
y = df_model["Exited"]
X = df_model.drop("Exited", axis=1)

#target feature ayrımı yaptım

X.shape, y.shape

((10000, 11), (10000,))

In [33]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y  #imbalanced data için stratify kullanılır
)


In [34]:
#Feature Scaling

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#scaling yani ölçekleme yaptım böylece sayısal featurelar aynı ölçeğe gelmiş oldu
#yani büyük sayılar ile küçük sayılar aynı seviyeye gelmiş oldu
#bu da modelin performansını artırır -> modelin büyük sayılara daha çok önem vermesini engeller

In [36]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

LogisticRegression()

In [42]:
y_pred = log_model.predict(X_test_scaled)
y_pred[0:10]  #y_pred bir numpy array olduğu için head kullanılmaz

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])